# SDXL + OpenPose: 이미지 품질 개선 실험

**별도 실험 초안 · Colab GPU 실행 검증 전.** 기존 과제 노트와 결과를 바꾸지 않습니다. 512px SD 1.5 결과와 1024px SDXL 결과를 비교하되 모델과 해상도가 함께 바뀌므로 차이의 단일 원인을 확정할 수 없습니다. 얼굴·팔 개선은 보장되지 않습니다.

[SDXL 기본 모델](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0)과 [SDXL용 OpenPose ControlNet](https://huggingface.co/xinsir/controlnet-openpose-sdxl-1.0)을 사용합니다. 각 모델·포즈 추출 도구의 라이선스와 이미지 권리를 공개 전 확인하세요.

**준비:** 새 Colab 세션에서 GPU(T4 등)를 선택하고 사용 권리가 있는 사진 한 장을 준비하세요. 모델 다운로드는 여러 GB이고 시간이 걸립니다. 무료 GPU·RAM·저장 공간과 실행 시간은 보장되지 않습니다. 메모리 오류가 나면 결과를 알리고 중단하세요. 이 노트에는 인페인팅은 아직 포함되지 않습니다.

## 1. GPU 확인과 패키지 설치
이전 노트와 같은 Gradio는 사용하지 않습니다. 설치 시 기존 패키지와 충돌 경고가 나올 수 있습니다. 설치가 실제 실패하면 오류 전체를 확인합니다.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('런타임 → 런타임 유형 변경 → GPU를 선택하세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('PyTorch:', torch.__version__)

In [ ]:
# 이전 실험에서 설치된 조합을 참고합니다. Colab 버전에 따라 호환성 확인이 필요해요.
%pip install 'diffusers==0.39.0' 'transformers==4.57.6' 'accelerate' 'controlnet-aux==0.0.10' 'huggingface-hub==0.36.2' safetensors

## 2. 포즈 사진 업로드
파일 선택 창에서 한 장을 선택합니다. 공개 가능한 사진만 사용하세요. 1024px 정사각형에 비율을 유지해 넣으므로 사진이 잘리지 않지만 여백이 생깁니다.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image, ImageOps
from IPython.display import display

uploaded = files.upload()
names = list(uploaded)
if len(names) != 1:
    raise ValueError('사진을 정확히 한 장 선택하고 이 셀을 다시 실행하세요.')
out = Path('/content/sdxl_pose_experiment')
out.mkdir(parents=True, exist_ok=True)
original = ImageOps.exif_transpose(Image.open(names[0])).convert('RGB')
reference = ImageOps.pad(original, (1024, 1024), method=Image.Resampling.LANCZOS, color='white')
reference.save(out / 'reference.png')
print('입력:', names[0])
display(reference)

## 3. 관절 추출 및 확인
기본 OpenPose 전처리기로 뼈대를 만들고 원본과 비교합니다. 팔이 빠지거나 엉킨 경우 생성 결과도 그 조건에 영향을 받을 수 있습니다.

In [ ]:
from controlnet_aux import OpenposeDetector
import gc
detector = OpenposeDetector.from_pretrained('lllyasviel/ControlNet')
skeleton = detector(reference, detect_resolution=512, image_resolution=1024).resize((1024, 1024))
skeleton.save(out / 'skeleton.png')
display(reference, skeleton)
del detector
gc.collect()
torch.cuda.empty_cache()

## 4. SDXL과 SDXL용 ControlNet 준비
모델 카드의 호환 예제를 따르며 CPU 오프로딩과 VAE 타일링으로 메모리 사용을 줄입니다. 장치와 Colab 환경에 따라 메모리가 부족하거나 느릴 수 있습니다. 다운로드 또는 모델 접근 오류가 나면 오류를 확인하세요.

In [ ]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
base_model = 'stabilityai/stable-diffusion-xl-base-1.0'
# Diffusers 형식의 safetensors 가중치가 있는 SDXL용 포즈 모델
control_model = 'xinsir/controlnet-openpose-sdxl-1.0'
controlnet = ControlNetModel.from_pretrained(control_model, torch_dtype=torch.float16, use_safetensors=True)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    base_model, controlnet=controlnet, torch_dtype=torch.float16, use_safetensors=True
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_tiling()
print('모델 준비 완료')

## 5. 전신 이미지 한 장 생성
실험은 한 장만 먼저 생성해 메모리와 결과를 확인합니다. 이전 실험과 비슷한 문장을 사용합니다. 완료 후 원본·뼈대·결과를 직접 비교하고, 오류나 추가 팔이 있는지 확인하세요.

In [ ]:
import json, platform
from importlib.metadata import version

prompt = 'a single dancer in casual clothes in a bright dance studio, full body, photograph'
negative_prompt = 'extra limbs, extra arms, deformed anatomy, blurry face'
seed, steps, control_strength = 42, 30, 0.8
result = pipe(prompt=prompt, negative_prompt=negative_prompt, image=skeleton,
              width=1024, height=1024, num_inference_steps=steps,
              controlnet_conditioning_scale=control_strength, guidance_scale=7.5,
              generator=torch.Generator(device='cpu').manual_seed(seed)).images[0]
result.save(out / 'result_sdxl.png')
settings = {'prompt':prompt, 'negative_prompt':negative_prompt, 'seed':seed, 'steps':steps,
            'control_strength':control_strength, 'guidance_scale':7.5, 'size':[1024,1024],
            'base_model':base_model, 'control_model':control_model,
            'gpu':torch.cuda.get_device_name(0), 'torch':torch.__version__,
            'diffusers':version('diffusers'), 'transformers':version('transformers'),
            'huggingface_hub':version('huggingface-hub'), 'input_file':names[0]}
(out / 'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')
display(result)
print('이미지와 설정 저장:', out)

## 6. 결과 다운로드와 관찰
ZIP에는 참조 사진·뼈대·결과와 설정이 들어 있습니다. 공개하기 전에 사진 사용 권리와 개인 정보를 확인합니다. 같은 이름의 이전 ZIP은 다시 만들면 덮어쓰니 다운로드 후 이름을 바꿔 보관하세요.

In [ ]:
import shutil
archive = shutil.make_archive('/content/sdxl_pose_results', 'zip', out)
files.download(archive)

## 관찰 기록 — 실행 후 직접 작성
- 포즈와 팔 개수: 아직 확인 전
- 얼굴 표현: 아직 확인 전
- SD 1.5 결과와 차이: 아직 확인 전 (모델과 해상도가 함께 다름)
- 오류·실행 시간·GPU: 아직 확인 전
- 이후 얼굴·팔 인페인팅 필요 여부: 아직 확인 전